# Architecture

## Tokenisation

In [7]:
import glob

corpus_files = sorted(glob.glob(os.path.join("..", "data", "clean_books", "*.txt")))
print(f"Training on {len(corpus_files)} files")

tokenizer.train(files=corpus_files, trainer=trainer)

tokenizer_path = os.path.join("..", "data", "tokenizer.json")
tokenizer.save(tokenizer_path)

print(f"Trained vocab size: {tokenizer.get_vocab_size()}")
print(f"Saved to {tokenizer_path}")

Training on 50 files



Trained vocab size: 8000
Saved to ../data/tokenizer.json


### Why save it to disk?

Training took a few seconds here, but the point stands at any scale: you want to train a tokenizer once and reuse the exact same one everywhere — for every training run, every evaluation, and eventually inference. If the vocabulary or merge rules ever drifted between runs, token ids would mean different things at different times, silently corrupting anything trained on top of it. Saving to a single JSON file (vocabulary + merges + configuration all together) means loading it back is a one-liner, and it's the artifact you'd version alongside the model itself, rather than something regenerated ad hoc.

### Loading it back, and what to expect

From here on, treat `../data/tokenizer.json` as the source of truth — reload it fresh rather than reusing the in-memory `tokenizer` object, exactly as a training script would when it starts up later with nothing but this file.

Try the same sentence as before: *"Also sprach Zarathustra: the eternal recurrence of the same."* A pretrained GPT-2 tokenizer splits `Zarathustra` into four pieces (`ĠZar`, `ath`, `ust`, `ra`), because it's a rare word in general web text. This corpus is titled after it and mentions it constantly — worth seeing whether that frequency earned it a token of its own.

In [9]:
loaded_tokenizer = Tokenizer.from_file(tokenizer_path)

sample = "Also sprach Zarathustra: the eternal recurrence of the same."
encoding = loaded_tokenizer.encode(sample)

print(f"{len(sample)} characters -> {len(encoding.ids)} tokens\n")
print("ids:   ", encoding.ids)
print("pieces:", encoding.tokens)
print("\ndecoded round-trip:", loaded_tokenizer.decode(encoding.ids))
print("round-trip matches original:", loaded_tokenizer.decode(encoding.ids) == sample)

60 characters -> 13 tokens

ids:    [7220, 4431, 572, 2239, 29, 178, 2617, 6057, 2641, 184, 178, 512, 17]
pieces: ['ĠAlso', 'Ġspr', 'ach', 'ĠZarathustra', ':', 'Ġthe', 'Ġeternal', 'Ġrecur', 'rence', 'Ġof', 'Ġthe', 'Ġsame', '.']

decoded round-trip:  Also sprach Zarathustra: the eternal recurrence of the same.
round-trip matches original: False


### Why look at token counts across the whole corpus?

Same reasoning as with characters earlier in `pre_processing.ipynb`: token count is what determines how many training sequences you actually get out of this corpus and how much training costs. Two things specific to a custom-trained tokenizer worth knowing:

- **A smaller vocabulary compresses text slightly less well.** A 50k-vocabulary tokenizer has more room to fold long chunks into single tokens; expect this 8k-vocabulary version to produce a somewhat lower characters-per-token ratio — that's the price of the smaller, corpus-appropriate embedding table from earlier.
- **Padding isn't configured yet.** `<pad>` was reserved as a special token during training, but a raw `Tokenizer` object doesn't pad automatically — that needs `tokenizer.enable_padding(pad_id=..., pad_token="<pad>")` explicitly, which is a decision for whenever we get to batching variable-length sequences for training, not now.

In [5]:
import json

import pandas as pd

metadata_path = os.path.join("..", "data", "metadata.json")
clean_dir = os.path.join("..", "data", "clean_books")

with open(metadata_path, encoding="utf-8") as f:
    metadata = json.load(f)

token_rows = []
for book_id, info in metadata.items():
    if "clean_char_count" not in info:
        continue

    with open(os.path.join(clean_dir, info["filename"]), encoding="utf-8") as f:
        text = f.read()

    n_tokens = len(loaded_tokenizer.encode(text).ids)

    token_rows.append({
        "book_id": book_id,
        "title": info["title"],
        "char_count": info["clean_char_count"],
        "token_count": n_tokens,
        "chars_per_token": info["clean_char_count"] / n_tokens,
    })

token_df = pd.DataFrame(token_rows).set_index("book_id")

total_tokens = token_df["token_count"].sum()
avg_ratio = token_df["chars_per_token"].mean()

print(f"Total corpus size: {total_tokens:,} tokens across {len(token_df)} books")
print(f"Average compression: {avg_ratio:.2f} characters per token")
token_df.sort_values("token_count", ascending=False)

Total corpus size: 6,063,746 tokens across 50 books
Average compression: 3.94 characters per token


,title,char_count,token_count,chars_per_token
book_id,,,,
27942,"A System of Logic, Ratiocinative and Inductive",2447179,574411,4.260328
11100,History of Modern Philosophy\nFrom Nicolas of ...,1502373,383157,3.921038
35722,"Ontology, or the Theory of Being",1260487,318159,3.961815
1497,The Republic,1194387,310443,3.847363
4705,A Treatise of Human Nature,1323609,308195,4.294713
4280,The Critique of Pure Reason,1269898,281957,4.503871
26495,"A System of Logic, Ratiocinative and Inductive...",1114477,261507,4.261748
6798,Aesthetical Essays of Friedrich Schiller,835338,200257,4.171330
852,Democracy and Education: An Introduction to th...,829436,190530,4.353309


# Tokeniser Class

In [ ]:
import os
import glob

from tokenizers import decoders
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

class TokeniseBooks:
    def __init__(self, cleaned_books_path, Vocab_size = 8000):
        self.books_path = cleaned_books_path
        self.vocab_size = Vocab_size
        self.tokeniser_path = None

    def train_tokeniser(self):

        tokeniser = Tokenizer(BPE(unk_token="<unk>"))
        tokeniser.pre_tokenizer = ByteLevel(add_prefix_space=True)
        tokeniser.decoder = decoders.ByteLevel()

        trainer = BpeTrainer(
            vocab_size=self.vocab_size,
            min_frequency=2,
            special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
        )
        corpus_files = sorted(glob.glob(os.path.join(self.books_path, "*.txt")))
        print(f"Training on {len(corpus_files)} files")
        tokeniser.train(files=corpus_files, trainer=trainer)
        tokeniser_path = os.path.join("..", "data", "tokenizer.json")
        tokeniser.save(tokeniser_path)

        print(f"Trained vocab size: {tokeniser.get_vocab_size()}")
        print(f"Saved to {tokeniser_path}")
        self.tokeniser_path = tokeniser_path

    def load_tokeniser(self):
        if self.tokeniser_path is None:
            print("Tokeniser path not set")
            return
        else:
            return Tokenizer.from_file(self.tokeniser_path)

    def test_tokeniser(self, tokeniser: Tokenizer, sample = "Also sprach Zarathustra: the eternal recurrence of the same."):
        loaded_tokeniser = tokeniser
        encoding = loaded_tokeniser.encode(sample)

        print(f"{len(sample)} characters -> {len(encoding.ids)} tokens\n")
        print("ids:   ", encoding.ids)
        print("pieces:", encoding.tokens)
        print("\ndecoded round-trip:", loaded_tokeniser.decode(encoding.ids))
        print("round-trip matches original:", loaded_tokeniser.decode(encoding.ids) == sample)
        

In [ ]:
loaded_tokenizer = Tokenizer.from_file(tokenizer_path)

sample = "Also sprach Zarathustra: the eternal recurrence of the same."
encoding = loaded_tokenizer.encode(sample)

print(f"{len(sample)} characters -> {len(encoding.ids)} tokens\n")
print("ids:   ", encoding.ids)
print("pieces:", encoding.tokens)
print("\ndecoded round-trip:", loaded_tokenizer.decode(encoding.ids))
print("round-trip matches original:", loaded_tokenizer.decode(encoding.ids) == sample)

In [3]:
import glob
import os

sorted(glob.glob(os.path.join("..", "data", "clean_books", "*.txt")))

['../data/clean_books/1016.txt',
 '../data/clean_books/11100.txt',
 '../data/clean_books/11224.txt',
 '../data/clean_books/11984.txt',
 '../data/clean_books/12004.txt',
 '../data/clean_books/1497.txt',
 '../data/clean_books/1635.txt',
 '../data/clean_books/1642.txt',
 '../data/clean_books/16833.txt',
 '../data/clean_books/1726.txt',
 '../data/clean_books/17556.txt',
 '../data/clean_books/19322.txt',
 '../data/clean_books/1998.txt',
 '../data/clean_books/22283.txt',
 '../data/clean_books/22364.txt',
 '../data/clean_books/23422.txt',
 '../data/clean_books/25012.txt',
 '../data/clean_books/25110.txt',
 '../data/clean_books/25172.txt',
 '../data/clean_books/2529.txt',
 '../data/clean_books/25447.txt',
 '../data/clean_books/26495.txt',
 '../data/clean_books/27597.txt',
 '../data/clean_books/27942.txt',
 '../data/clean_books/28696.txt',
 '../data/clean_books/31796.txt',
 '../data/clean_books/32168.txt',
 '../data/clean_books/32547.txt',
 '../data/clean_books/34901.txt',
 '../data/clean_books